# RAG 기본 구조 이해하기

## 1. 사전작업(Pre-processing) - 1~4 단계

![rag-1.png](./assets/rag-1.png)

![rag-1-graphic](./assets/rag-graphic-1.png)

사전 작업 단계에서는 데이터 소스를 Vector DB (저장소) 에 문서를 로드-분할-임베딩-저장 하는 4단계를 진행합니다.

- 1단계 문서로드(Document Load): 문서 내용을 불러옵니다.
- 2단계 분할(Text Split): 문서를 특정 기준(Chunk) 으로 분할합니다.
- 3단계 임베딩(Embedding): 분할된(Chunk) 를 임베딩하여 저장합니다.
- 4단계 벡터DB 저장: 임베딩된 Chunk 를 DB에 저장합니다.

## 2. RAG 수행(RunTime) - 5~8 단계

![rag-2.png](./assets/rag-2.png)

![](./assets/rag-graphic-2.png)

- 5단계 검색기(Retriever): 쿼리(Query) 를 바탕으로 DB에서 검색하여 결과를 가져오기 위하여 리트리버를 정의합니다. 리트리버는 검색 알고리즘이며(Dense, Sparse) 리트리버로 나뉘게 됩니다. 
  - **Dense**: 유사도 기반 검색(FAISS, DPR)
  - **Sparse**: 키워드 기반 검색(BM25, TF-IDF)
- 6단계 프롬프트: RAG 를 수행하기 위한 프롬프트를 생성합니다. 프롬프트의 context 에는 문서에서 검색된 내용이 입력됩니다. 프롬프트 엔지니어링을 통하여 답변의 형식을 지정할 수 있습니다.
- 7단계 LLM: 모델을 정의합니다.(GPT-3.5, GPT-4, Claude, etc..)
- 8단계 Chain: 프롬프트 - LLM - 출력 에 이르는 체인을 생성합니다.

## 환경설정


API KEY 를 설정합니다.


In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

LangChain으로 구축한 애플리케이션은 여러 단계에 걸쳐 LLM 호출을 여러 번 사용하게 됩니다. 이러한 애플리케이션이 점점 더 복잡해짐에 따라, 체인이나 에이전트 내부에서 정확히 무슨 일이 일어나고 있는지 조사할 수 있는 능력이 매우 중요해집니다. 이를 위한 최선의 방법은 [LangSmith](https://smith.langchain.com)를 사용하는 것입니다.

LangSmith가 필수는 아니지만, 유용합니다. LangSmith를 사용하고 싶다면, 위의 링크에서 가입한 후, 로깅 추적을 시작하기 위해 환경 변수를 설정해야 합니다.


In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH12-RAG")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH12-RAG


## 네이버 뉴스 기반 QA(Question-Answering) 챗봇

이번 튜토리얼에는 네이버 뉴스기사의 내용에 대해 질문할 수 있는 **뉴스기사 QA 앱** 을 구축할 것입니다.

이 가이드에서는 OpenAI 챗 모델과 임베딩, 그리고 Chroma 벡터 스토어를 사용할 것입니다.

먼저 다음의 과정을 통해 간단한 인덱싱 파이프라인과 RAG 체인을 약 20줄의 코드로 구현할 수 있습니다.

라이브러리

- `bs4`는 웹 페이지를 파싱하기 위한 라이브러리입니다.
- `langchain`은 AI와 관련된 다양한 기능을 제공하는 라이브러리로, 여기서는 특히 텍스트 분할(`RecursiveCharacterTextSplitter`), 문서 로딩(`WebBaseLoader`), 벡터 저장(`Chroma`, `FAISS`), 출력 파싱(`StrOutputParser`), 실행 가능한 패스스루(`RunnablePassthrough`) 등을 다룹니다.
- `langchain_openai` 모듈을 통해 OpenAI의 챗봇(`ChatOpenAI`)과 임베딩(`OpenAIEmbeddings`) 기능을 사용할 수 있습니다.


In [3]:
import bs4
from langchain import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


웹 페이지의 내용을 로드하고, 텍스트를 청크로 나누어 인덱싱하는 과정을 거친 후, 관련된 텍스트 스니펫을 검색하여 새로운 내용을 생성하는 과정을 구현합니다.

`WebBaseLoader`는 지정된 웹 페이지에서 필요한 부분만을 파싱하기 위해 `bs4.SoupStrainer`를 사용합니다.

[참고]

- `bs4.SoupStrainer` 는 편리하게 웹에서 원하는 요소를 가져올 수 있도록 해줍니다.

(예시)

```python
bs4.SoupStrainer(
    "div",
    attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
)
```


In [4]:
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000455038",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs

문서의 수: 1


[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000455038'}, page_content='\n저수지 바닥 드러난 강릉 "시간·격일제 급수 가능성, 생수 배부"\n\n\n\n\n\n\n1일 오후 강릉시민들이 맨바닥을 드러낸 오봉저수지를 근심스럽게 바라보고 있다. 〈사진=연합뉴스〉 최악의 가뭄을 겪고 있는 강원 강릉시가 저수율이 10% 미만으로 떨어지면 격일제·시간제 급수를 검토하는 등 강력한 대책을 시행하겠다고 밝혔습니다.  김홍규 강릉시장은 오늘(1일) 강릉시청 재난상황실에서 가뭄 대응 비상대책 2차 기자회견을 열고 향후 대책 방안을 발표했습니다.  강릉의 주요 식수원인 오봉저수지는 이날 기준으로 저수율이 14%대까지 떨어졌습니다. 예년 이맘때 저수율인 71.7%와 비교하면 4분의 1 수준입니다.  이로 인해 시민들이 사용하는 생활용수와 농업용수의 공급에 큰 어려움이 있는 상황입니다.  정부는 지난달 30일 자연재난으로는 처음으로 강릉을 재난사태 지역으로 선포하기도 했습니다.  김 시장은 "현재 소방차량을 포함한 71대를 투입해 연곡정수장 및 인근 지자체에서 홍제정수장으로 하루 2130톤(t)의 정수를 운반하고 있다"며 "저수율이 0%에 도달하는 최악의 상황이 오면 홍제정수장 전 구역에 대해 차량을 통한 운반급수를 실시할 계획"이라고 말했습니다.  이어 "최대 하루 400대의 살수차를 동원해 지방하천과 저수지 22개소에서 하루 1만5600톤의 원수를 오봉저수지에 추가로 투입할 계획"이라고 설명했습니다.  강릉시는 제한급수 조치도 강화하겠다는 방침입니다.  김 시장은 "오늘부터는 홍제정수장 정수구역 내 5만 3000여 수용가에 대해 계량기 75% 조절을 전면 시행한다"며 "저수율이 10% 미만으로 떨어지면 시간제·격일제 급수를 검토해 시행하겠다"고 밝혔습니다.  이미 계량기 50%를 잠그는 제한 급수를 시행해 왔는데, 더 강력한 조치에 들어가는 겁니다.  다만 의료시설과 사회복지시

`RecursiveCharacterTextSplitter`는 문서를 지정된 크기의 청크로 나눕니다.


In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = text_splitter.split_documents(docs)
len(splits)

4

`FAISS` 혹은 `Chroma`와 같은 vectorstore는 이러한 청크를 바탕으로 문서의 벡터 표현을 생성합니다.


In [6]:
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

`vectorstore.as_retriever()`를 통해 생성된 검색기는 `hub.pull`로 가져온 프롬프트와 `ChatOpenAI` 모델을 사용하여 새로운 내용을 생성합니다.

마지막으로, `StrOutputParser`는 생성된 결과를 문자열로 파싱합니다.


In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#Question: 
{question} 

#Context: 
{context} 

#Answer:"""
)

hub 에서 `teddynote/rag-prompt-korean` 프롬프트를 다운로드 받아 입력할 수 있습니다. 이런 경우 별도의 프롬프트 작성과정이 생략됩니다.

In [ ]:
# prompt = hub.pull("teddynote/rag-prompt-korean")
# prompt

In [8]:
llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)


# 체인을 생성합니다.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

스트리밍 출력을 위하여 `stream_response` 를 사용합니다.

In [9]:
from langchain_teddynote.messages import stream_response

> [LangSmith Trace 보기](https://smith.langchain.com/public/c6047a61-8f44-48e5-89eb-b1e8a6321cea/r)


In [10]:
answer = rag_chain.stream("최악의 가뭄을 겪고 있는 강원 강릉시가 내놓은 대첵은?")
stream_response(answer)

최악의 가뭄을 겪고 있는 강원 강릉시는 다음과 같은 대책을 내놓았습니다.

- 저수율이 10% 미만으로 떨어지면 시간제·격일제 급수를 검토·시행
- 계량기 조절을 강화해 홍제정수장 정수구역 내 5만 3000여 수용가에 대해 계량기 75% 조절 전면 시행
- 농업용수 공급 중단 요청 및 저수지와 지방하천을 활용한 대체용수 긴급 지원 계획
- 수영장, 사우나 등 비필수 물 사용 시설 운영 제한 및 숙박률 조정 요청
- 강릉관광개발공사 운영 숙박시설은 저수율 10% 미만 시 운영 전면 중단
- 하루 최대 400대 살수차 동원해 지방하천과 저수지 22개소에서 원수 1만5600톤 오봉저수지에 추가 투입
- 소방차량 포함 71대 투입해 연곡정수장 및 인근 지자체에서 홍제정수장으로 하루 2130톤 정수 운반
- 차량을 통한 운반급수 계획(저수율 0% 도달 시)
- 생수 200만 병 확보 목표(현재 135만 병 확보), 교육·사회복지시설에 우선 배부, 이후 전 시민에게 확대 예정
- 노후 상수관망 현대화, 연곡정수장 정비, 지하수 저류댐 설치 등 생활용수·농업용수 관리 사업 추진

이와 같이 강릉시는 제한급수 강화, 대체용수 확보, 생수 배부, 시설 운영 제한 등 다각적인 가뭄 대응 비상대책을 시행하고 있습니다.

> [LangSmith Trace 보기](https://smith.langchain.com/public/ed21d80e-b4da-4a08-823b-ed980db9c347/r)


In [12]:
answer = rag_chain.stream("오봉저수지의 예년 이맘때 수위는 어느 정도인가요?")
stream_response(answer)

오봉저수지의 예년 이맘때 수위는 저수율 기준으로 약 71.7% 정도입니다.

> [LangSmith Trace 보기](https://smith.langchain.com/public/df80c528-61d6-4c83-986a-3373a4039dae/r)


In [13]:
answer = rag_chain.stream("강릉시는 제한급수 조치도 강화하겠다는 방침인데 예외 시설들은 어디인가?")
stream_response(answer)

강릉시가 제한급수 조치를 강화하는 가운데, 예외 시설은 의료시설, 사회복지시설, 교정시설 등 필수 시설입니다. 이들 시설에는 예외 없이 생활용수를 공급하기 위해 하루 20대의 살수차를 전담 배치한다고 합니다.

> [LangSmith Trace 보기](https://smith.langchain.com/public/1a613ee7-6eaa-482f-a45f-8c22b4e60fbf/r)


In [15]:
answer = rag_chain.stream("생수 배부와 관련해서는 최소 2리터(ℓ) 기준 몇병을 확보하고 있는가?")
stream_response(answer)

생수 배부와 관련해서는 최소 2리터(ℓ) 기준으로 200만 병 확보를 목표로 하고 있으며, 현재 135만 병을 확보한 상태입니다.